# Datalake Companies Investigation

Investigating the datasets:
- `input/cvm-companies-registration`
- `input/b3-company-details`

Goal: Create a consolidated company dataset in `staging` with relevant information.

In [1]:
import os
from pathlib import Path

import duckdb
import pandas as pd

# Get datalake path
datalake_path = os.environ.get("BRASA_DATA_PATH", "/home/wilson/snap/brasa")
input_path = os.path.join(datalake_path, "db", "input")
staging_path = os.path.join(datalake_path, "db", "staging")

print(f"Datalake path: {datalake_path}")
print(f"Input path: {input_path}")
print(f"Staging path: {staging_path}")
print(
    f"\nDirectory exists: input={os.path.exists(input_path)}, staging={os.path.exists(staging_path)}"
)

Datalake path: /home/wilson/snap/brasa/
Input path: /home/wilson/snap/brasa/db/input
Staging path: /home/wilson/snap/brasa/db/staging

Directory exists: input=True, staging=True


## 1. Connect to DuckDB and Explore CVM Companies Registration

In [2]:
# Connect to DuckDB
conn = duckdb.connect(":memory:")

# Enable httpfs extension for reading from various file formats
conn.execute("INSTALL httpfs; LOAD httpfs")

# Load CVM companies registration dataset
cvm_path = os.path.join(input_path, "cvm-companies-registration")
print(f"CVM Companies Registration path: {cvm_path}")
print("\nDirectory structure:")
for root, dirs, files in os.walk(cvm_path):
    level = root.replace(cvm_path, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files[:5]:  # Show first 5 files
        print(f"{subindent}{file}")

CVM Companies Registration path: /home/wilson/snap/brasa/db/input/cvm-companies-registration

Directory structure:
cvm-companies-registration/
  refdate=2026-02-14/
    8eee544f0073484d8923499e824af02e-0.parquet


In [ ]:
# Read CVM companies registration parquet files with Hive partitioning
cvm_query = (
    f"SELECT * FROM read_parquet('{cvm_path}/**/*.parquet', hive_partitioning=true)"
)
cvm_df = conn.execute(cvm_query).fetch_df()

print("CVM Companies Registration Dataset")
print(f"Shape: {cvm_df.shape}")
print(f"\nColumns: {list(cvm_df.columns)}")
print(f"\nData Types:\n{cvm_df.dtypes}")
print("\nFirst few rows:")
display(cvm_df.head())

CVM Companies Registration Dataset
Shape: (2654, 48)

Columns: ['cnpj_cia', 'denom_social', 'denom_comerc', 'dt_reg', 'dt_const', 'dt_cancel', 'motivo_cancel', 'sit', 'dt_ini_sit', 'code_cvm', 'setor_ativ', 'tp_merc', 'categ_reg', 'dt_ini_categ', 'sit_emissor', 'dt_ini_sit_emissor', 'controle_acionario', 'tp_ender', 'logradouro', 'compl', 'bairro', 'mun', 'uf', 'pais', 'cep', 'ddd_tel', 'tel', 'ddd_fax', 'fax', 'email', 'tp_resp', 'resp', 'dt_ini_resp', 'logradouro_resp', 'compl_resp', 'bairro_resp', 'mun_resp', 'uf_resp', 'pais_resp', 'cep_resp', 'ddd_tel_resp', 'tel_resp', 'ddd_fax_resp', 'fax_resp', 'email_resp', 'cnpj_auditor', 'auditor', 'refdate']

Data Types:
cnpj_cia                      object
denom_social                  object
denom_comerc                  object
dt_reg                datetime64[us]
dt_const              datetime64[us]
dt_cancel             datetime64[us]
motivo_cancel                 object
sit                           object
dt_ini_sit            datetim

,cnpj_cia,denom_social,denom_comerc,dt_reg,dt_const,dt_cancel,motivo_cancel,sit,dt_ini_sit,code_cvm,...,pais_resp,cep_resp,ddd_tel_resp,tel_resp,ddd_fax_resp,fax_resp,email_resp,cnpj_auditor,auditor,refdate
0,08.773.135/0001-00,2W ECOBANK S.A. - EM RECUPERAÇÃO JUDICIAL,2W ECOBANK S.A.,2020-10-29,2007-03-23,NaT,None,ATIVO,2020-10-29,25224,...,None,4711905,11,39579400,None,None,juridico@2wecobank.com.br,10.830.108/0001-65,GRANT THORNTON AUDITORES INDEPENDENTES LTDA.,2026-02-14
1,11.396.633/0001-87,3A COMPANHIA SECURITIZADORA,TRIPLO A COMPANHIA SECURITIZADORA,2010-03-08,2009-11-03,2015-12-18,Cancelamento Voluntário - IN CVM 480/09,CANCELADA,2015-12-18,21954,...,None,20020000,21,22338867,21,22338867,juridico@triploasec.com.br,60.525.706/0001-07,MOORE STEPHENS LIMA LUCCHESI AUDITORES INDEPEN...,2026-02-14
2,01.547.749/0001-16,521 PARTICIPAÇOES S.A. - EM LIQUIDAÇÃO EXTRAJU...,521 PARTICIPAÇÕES S/A,1997-07-11,1996-07-30,NaT,None,ATIVO,1997-07-11,16330,...,None,None,None,None,None,None,None,02.878.522/0001-16,IRKO HIRASHIMA AUDITORES INDEPENDENTES LTDA.,2026-02-14
3,01.851.771/0001-55,524 PARTICIPAÇOES SA,524 PARTICIPACOES SA,1997-05-30,1997-04-02,NaT,None,ATIVO,1997-05-30,16284,...,None,20020010,21,38043700,None,None,gar@opportunity.com.br,10.830.108/0001-65,GRANT THORNTON AUDITORES INDEPENDENTES LTDA.,2026-02-14
4,01.919.008/0001-19,525 PARTICIPAÇOES SA,525 PARTICIPACOES SA,1997-07-16,1997-04-02,2006-05-30,ELISÃO POR INCORPORAÇÃO,CANCELADA,2006-05-30,16349,...,None,22290160,21,21967200,21,21967201,kevin@mattosfilho.com.br,11.245.719/0001-09,DIRECTA AUDITORES,2026-02-14


In [4]:
# Check for duplicates and missing values
print("CVM Dataset Info:")
print(f"Total rows: {len(cvm_df)}")
print("\nMissing values:")
print(cvm_df.isnull().sum())
print("\nUnique values for key columns:")
for col in ["cnpj", "company_name", "trading_name", "company_status"]:
    if col in cvm_df.columns:
        print(f"{col}: {cvm_df[col].nunique()} unique values")

CVM Dataset Info:
Total rows: 2654

Missing values:
cnpj_cia                 0
denom_social             0
denom_comerc            93
dt_reg                   0
dt_const               929
dt_cancel              765
motivo_cancel          767
sit                      0
dt_ini_sit               0
code_cvm                 0
setor_ativ               6
tp_merc               1985
categ_reg             1239
dt_ini_categ          1239
sit_emissor           1330
dt_ini_sit_emissor    1330
controle_acionario       2
tp_ender                 1
logradouro               1
compl                 1209
bairro                 645
mun                      1
uf                       1
pais                    93
cep                      2
ddd_tel                 26
tel                     26
ddd_fax                859
fax                    846
email                 1050
tp_resp                 70
resp                    70
dt_ini_resp             70
logradouro_resp         76
compl_resp            1279
bai

## 2. Explore B3 Company Details

In [5]:
# Load B3 company details dataset
b3_path = os.path.join(input_path, "b3-company-details")
print(f"B3 Company Details path: {b3_path}")
print("\nSample directory structure:")
for i, dir_name in enumerate(os.listdir(b3_path)[:5]):
    print(f"  {dir_name}/")

B3 Company Details path: /home/wilson/snap/brasa/db/input/b3-company-details

Sample directory structure:
  codeCVMarg=20532/
  codeCVMarg=910075/
  codeCVMarg=27138/
  codeCVMarg=27863/
  codeCVMarg=8192/


In [ ]:
# Read B3 company details parquet files with Hive partitioning
b3_query = (
    f"SELECT * FROM read_parquet('{b3_path}/**/*.parquet', hive_partitioning=true)"
)
b3_df = conn.execute(b3_query).fetch_df()

print("B3 Company Details Dataset")
print(f"Shape: {b3_df.shape}")
print(f"\nColumns: {list(b3_df.columns)}")
print(f"\nData Types:\n{b3_df.dtypes}")
print("\nFirst few rows:")
display(b3_df.head())

B3 Company Details Dataset
Shape: (29032, 24)

Columns: ['issuingCompany', 'companyName', 'tradingName', 'cnpj', 'industryClassification', 'industryClassificationEng', 'activity', 'website', 'hasQuotation', 'status', 'marketIndicator', 'market', 'institutionCommon', 'institutionPreferred', 'code', 'codeCVM', 'lastDate', 'hasEmissions', 'hasBDR', 'typeBDR', 'describleCategoryBVMF', 'isin', 'codeCVMarg', 'refdate']

Data Types:
issuingCompany                       object
companyName                          object
tradingName                          object
cnpj                                 object
industryClassification               object
industryClassificationEng            object
activity                             object
website                              object
hasQuotation                         object
status                               object
marketIndicator                      object
market                               object
institutionCommon                    objec

,issuingCompany,companyName,tradingName,cnpj,industryClassification,industryClassificationEng,activity,website,hasQuotation,status,...,code,codeCVM,lastDate,hasEmissions,hasBDR,typeBDR,describleCategoryBVMF,isin,codeCVMarg,refdate
0,BBAS,BCO BRASIL S.A.,BRASIL,00000000000191,Financeiro / Intermediários Financeiros / Bancos,None,Banco Múltiplo,www.bb.com.br,None,A,...,BBAS11,1023,16/01/2024 23:12:26,False,False,None,None,BRBBASA04OR8,1023,2024-01-17
1,BBAS,BCO BRASIL S.A.,BRASIL,00000000000191,Financeiro / Intermediários Financeiros / Bancos,None,Banco Múltiplo,www.bb.com.br,None,A,...,BBAS12,1023,16/01/2024 23:12:26,False,False,None,None,BRBBASA05OR5,1023,2024-01-17
2,BBAS,BCO BRASIL S.A.,BRASIL,00000000000191,Financeiro / Intermediários Financeiros / Bancos,None,Banco Múltiplo,www.bb.com.br,None,A,...,BBAS3,1023,16/01/2024 23:12:26,False,False,None,None,BRBBASACNOR3,1023,2024-01-17
3,BBAS,BCO BRASIL S.A.,BRASIL,00000000000191,Financeiro / Intermediários Financeiros / Bancos,None,Banco Múltiplo,www.bb.com.br,None,A,...,BBAS11,1023,19/01/2024 23:12:37,False,False,None,None,BRBBASA04OR8,1023,2024-01-20
4,BBAS,BCO BRASIL S.A.,BRASIL,00000000000191,Financeiro / Intermediários Financeiros / Bancos,None,Banco Múltiplo,www.bb.com.br,None,A,...,BBAS12,1023,19/01/2024 23:12:37,False,False,None,None,BRBBASA05OR5,1023,2024-01-20


In [7]:
# Check for duplicates and missing values
print("B3 Dataset Info:")
print(f"Total rows: {len(b3_df)}")
print("\nMissing values:")
print(b3_df.isnull().sum())
print("\nPartition columns info:")
if "refdate" in b3_df.columns:
    print(f"Unique refdates: {b3_df['refdate'].nunique()}")
    print(f"Date range: {b3_df['refdate'].min()} to {b3_df['refdate'].max()}")
if "codeCVMarg" in b3_df.columns:
    print(f"Unique codeCVMarg: {b3_df['codeCVMarg'].nunique()}")

B3 Dataset Info:
Total rows: 29032

Missing values:
issuingCompany                   0
companyName                      0
tradingName                      0
cnpj                             0
industryClassification           0
industryClassificationEng    29032
activity                      2649
website                          0
hasQuotation                 29032
status                           0
marketIndicator                  0
market                           0
institutionCommon                0
institutionPreferred             0
code                          2419
codeCVM                          0
lastDate                         0
hasEmissions                     0
hasBDR                           0
typeBDR                      29007
describleCategoryBVMF        29032
isin                          2419
codeCVMarg                       0
refdate                          0
dtype: int64

Partition columns info:
Unique refdates: 49
Date range: 2024-01-17 00:00:00 to 2026-02-14 00:0

## 3. Analyze Data Quality and Deduplication

In [8]:
# Check for duplicates in CVM data
print("CVM Duplicates Analysis:")
if "cnpj" in cvm_df.columns:
    print(f"Total rows: {len(cvm_df)}")
    print(f"Unique CNPJs: {cvm_df['cnpj'].nunique()}")
    print(f"Duplicate CNPJs: {len(cvm_df) - cvm_df['cnpj'].nunique()}")

    # Show duplicated CNPJs
    duplicated_cnpjs = cvm_df[cvm_df["cnpj"].duplicated(keep=False)].sort_values("cnpj")
    print("\nSample of duplicated records:")
    display(duplicated_cnpjs.head(10))

CVM Duplicates Analysis:


In [17]:
# Check for duplicates in B3 data
print("B3 Duplicates Analysis:")
if "refdate" in b3_df.columns and "company_name" in b3_df.columns:
    print(f"Total rows: {len(b3_df)}")
    print("\nRecords per company (unique company_name values):")
    company_counts = b3_df["company_name"].value_counts()
    print(company_counts.head(10))

    # Check if same company has multiple records due to different refdates
    print("\nSample of duplicated records across refdate:")
    if len(company_counts[company_counts > 1]) > 0:
        sample_company = company_counts.index[0]
        sample_records = b3_df[b3_df["company_name"] == sample_company].sort_values(
            "refdate"
        )
        display(
            sample_records[["company_name", "refdate", "asset_name", "cnpj"]].head(10)
        )

B3 Duplicates Analysis:


## 4. Create Consolidated Company Dataset

In [26]:
# Create consolidated dataset using improved query
# Filters: Only BOLSA (stock exchange) companies with latest CVM registration
consolidated_query = f"""
-- CVM Companies Registration data (deduplicated by CNPJ)
-- Only companies registered for BOLSA (Stock Exchange)
-- Use latest refdate
WITH cvm_latest AS (
    SELECT DISTINCT
        code_cvm,
        cnpj_cia as cnpj,
        denom_social as company_name,
        denom_comerc as trading_name,
        sit as company_status,
        ROW_NUMBER() OVER (PARTITION BY cnpj_cia ORDER BY refdate DESC) as rn
    FROM read_parquet('{cvm_path}/**/*.parquet', hive_partitioning=true)
    WHERE tp_merc = 'BOLSA'
    AND refdate = (SELECT MAX(refdate) FROM read_parquet('{cvm_path}/**/*.parquet', hive_partitioning=true))
),
-- B3 Company Details (deduplicated by codeCVM)
b3_latest AS (
    SELECT DISTINCT
        codeCVM as code_cvm,
        issuingCompany,
        companyName,
        cnpj,
        tradingName,
        isin,
        industryClassification as industry_sector,
        ROW_NUMBER() OVER (PARTITION BY codeCVMarg ORDER BY refdate DESC) as rn
    FROM read_parquet('{b3_path}/**/*.parquet', hive_partitioning=true)
)
SELECT
    CAST(cvm.code_cvm AS VARCHAR) as code_cvm,
    COALESCE(b3.companyName, cvm.company_name) as company_name,
    COALESCE(b3.tradingName, cvm.trading_name) as trading_name,
    b3.issuingCompany as asset_name,
    cvm.company_status,
    b3.isin,
    b3.industry_sector,
    cvm.cnpj as cvm_cnpj,
    b3.cnpj as b3_cnpj
FROM cvm_latest cvm
LEFT JOIN b3_latest b3 ON cvm.code_cvm = b3.code_cvm
WHERE (cvm.rn = 1 OR cvm.rn IS NULL) AND (b3.rn = 1 OR b3.rn IS NULL)
ORDER BY cvm.company_name
"""

consolidated_df = conn.execute(consolidated_query).fetch_df()

print("Consolidated Company Dataset (BOLSA - Stock Exchange Companies)")
print(f"Shape: {consolidated_df.shape}")
print(f"\nColumns: {list(consolidated_df.columns)}")
print("\nFirst few rows:")
display(consolidated_df.head(10))

Consolidated Company Dataset (BOLSA - Stock Exchange Companies)
Shape: (464, 9)

Columns: ['code_cvm', 'company_name', 'trading_name', 'asset_name', 'company_status', 'isin', 'industry_sector', 'cvm_cnpj', 'b3_cnpj']

First few rows:


,code_cvm,company_name,trading_name,asset_name,company_status,isin,industry_sector,cvm_cnpj,b3_cnpj
0,21725,ADVANCED DIGITAL HEALTH MEDICINA PREVENTIVA S.A.,HUB COSMÉTICOS S.A.,None,CANCELADA,None,None,10.345.009/0001-98,None
1,25283,AERIS IND. E COM. DE EQUIP. GERACAO DE ENERGIA...,AERIS,AERI,ATIVO,BRAERIACNOR4,Bens Industriais / Máquinas e Equipamentos / M...,12.528.708/0001-07,12528708000107
2,19313,AES ELPA SA,AES ELPA,None,CANCELADA,None,None,01.917.705/0001-30,None
3,18350,AES TIETE SA,AES TIETE SA,None,CANCELADA,None,None,02.998.609/0001-27,None
4,18970,AES TIETE ENERGIA SA,AES TIETE E,TIET,CANCELADA,None,Utilidade Pública / Energia Elétrica / Energia...,04.128.563/0001-10,04128563000110
5,20150,AFLUENTE GERAÇÃO DE ENERGIA ELÉTRICA S.A.,AFLUENTE G,None,CANCELADA,None,None,07.620.094/0001-40,None
6,22179,AFLUENTE TRANSMISSÃO DE ENERGIA ELÉTRICA S/A,AFLUENTE T,AFLT,ATIVO,BRAFLTACNOR1,Utilidade Pública / Energia Elétrica / Energia...,10.338.320/0001-00,10338320000100
7,21911,AGRE EMPREENDIMENTOS IMOBILIÁRIOS SA,AGRE EMPREENDIMENTOS IMOBILIÁRIOS SA,None,CANCELADA,None,None,11.040.082/0001-14,None
8,9954,ALFA HOLDINGS S.A.,ALFA HOLDING,RPAD,ATIVO,BRRPADACNOR1,Financeiro / Intermediários Financeiros / Bancos,17.167.396/0001-69,17167396000169
9,21300,ALIANSCE SHOPPING CENTERS S/A,ALIANSCE,None,CANCELADA,None,None,06.082.980/0001-03,None


In [27]:
# Check data quality
print("Consolidated Dataset Quality (BOLSA Companies):")
print(f"Total unique companies: {len(consolidated_df)}")
print("\nMissing values:")
print(consolidated_df.isnull().sum())
print("\nData types:")
print(consolidated_df.dtypes)
print("\nCompany Status Distribution:")
print(consolidated_df["company_status"].value_counts())

Consolidated Dataset Quality (BOLSA Companies):
Total unique companies: 464

Missing values:
code_cvm             0
company_name         0
trading_name         1
asset_name          97
company_status       0
isin               163
industry_sector     97
cvm_cnpj             0
b3_cnpj             97
dtype: int64

Data types:
code_cvm           object
company_name       object
trading_name       object
asset_name         object
company_status     object
isin               object
industry_sector    object
cvm_cnpj           object
b3_cnpj            object
dtype: object

Company Status Distribution:
company_status
ATIVO                        327
CANCELADA                    135
SUSPENSO(A) - DECISÃO ADM      2
Name: count, dtype: int64


## 5. Save Consolidated Dataset to Staging

In [28]:
# Prepare final consolidated dataset with requested fields only
final_df = consolidated_df[
    [
        "code_cvm",
        "company_name",
        "trading_name",
        "asset_name",
        "company_status",
        "isin",
        "industry_sector",
    ]
].copy()

# Create staging directory if it doesn't exist
companies_staging_path = os.path.join(staging_path, "companies")
os.makedirs(companies_staging_path, exist_ok=True)

# Save as parquet for better performance
consolidated_parquet = os.path.join(
    companies_staging_path, "consolidated_bolsa.parquet"
)
final_df.to_parquet(consolidated_parquet, index=False)

print(f"Consolidated dataset (BOLSA) saved to: {consolidated_parquet}")
print(f"File size: {os.path.getsize(consolidated_parquet) / 1024:.2f} KB")

# Also save as CSV for easier inspection
consolidated_csv = os.path.join(companies_staging_path, "consolidated_bolsa.csv")
final_df.to_csv(consolidated_csv, index=False)
print(f"\nAlso saved as CSV: {consolidated_csv}")
print(f"File size: {os.path.getsize(consolidated_csv) / 1024:.2f} KB")

Consolidated dataset (BOLSA) saved to: /home/wilson/snap/brasa/db/staging/companies/consolidated_bolsa.parquet
File size: 29.57 KB

Also saved as CSV: /home/wilson/snap/brasa/db/staging/companies/consolidated_bolsa.csv
File size: 52.14 KB


In [29]:
# Verify by reading back
verified_df = pd.read_parquet(consolidated_parquet)
print("Staging directory contents:")
for file in os.listdir(companies_staging_path):
    filepath = os.path.join(companies_staging_path, file)
    if os.path.isfile(filepath):
        size = os.path.getsize(filepath) / 1024
        print(f"  {file}: {size:.2f} KB")

print(f"\nVerification - Rows: {len(verified_df)}, Columns: {len(verified_df.columns)}")
print(f"Column names: {list(verified_df.columns)}")
print("\nData types:")
print(verified_df.dtypes)
print("\nFirst 5 rows:")
display(verified_df.head())

Staging directory contents:
  consolidated_with_classification.csv: 384.32 KB
  consolidated.csv: 287.35 KB
  consolidated.parquet: 154.04 KB
  consolidated_bolsa.parquet: 29.57 KB
  consolidated_with_classification.parquet: 163.86 KB
  consolidated_bolsa.csv: 52.14 KB

Verification - Rows: 464, Columns: 7
Column names: ['code_cvm', 'company_name', 'trading_name', 'asset_name', 'company_status', 'isin', 'industry_sector']

Data types:
code_cvm           object
company_name       object
trading_name       object
asset_name         object
company_status     object
isin               object
industry_sector    object
dtype: object

First 5 rows:


,code_cvm,company_name,trading_name,asset_name,company_status,isin,industry_sector
0,21725,ADVANCED DIGITAL HEALTH MEDICINA PREVENTIVA S.A.,HUB COSMÉTICOS S.A.,None,CANCELADA,None,None
1,25283,AERIS IND. E COM. DE EQUIP. GERACAO DE ENERGIA...,AERIS,AERI,ATIVO,BRAERIACNOR4,Bens Industriais / Máquinas e Equipamentos / M...
2,19313,AES ELPA SA,AES ELPA,None,CANCELADA,None,None
3,18350,AES TIETE SA,AES TIETE SA,None,CANCELADA,None,None
4,18970,AES TIETE ENERGIA SA,AES TIETE E,TIET,CANCELADA,None,Utilidade Pública / Energia Elétrica / Energia...


## 6. Summary and Statistics

In [30]:
print("=" * 70)
print("DATALAKE INVESTIGATION SUMMARY - BOLSA (STOCK EXCHANGE) COMPANIES")
print("=" * 70)

print("\n1. SOURCE DATASETS:")
print(f"   - CVM Companies Registration: {len(cvm_df)} records (all)")
print(
    f"   - CVM BOLSA filter: {len(consolidated_df)} companies registered for stock exchange"
)
print(f"   - B3 Company Details: {len(b3_df)} records (all)")

print("\n2. CONSOLIDATED DATASET (BOLSA):")
print(f"   - Total unique companies: {len(consolidated_df)}")
print(f"   - Columns: {len(consolidated_df.columns)}")

print("\n3. DATA COVERAGE:")
print(f"   - Companies with code_cvm: {consolidated_df['code_cvm'].notna().sum()}")
print(
    f"   - Companies with company_name: {consolidated_df['company_name'].notna().sum()}"
)
print(
    f"   - Companies with trading_name: {consolidated_df['trading_name'].notna().sum()}"
)
print(f"   - Companies with asset_name: {consolidated_df['asset_name'].notna().sum()}")
print(f"   - Companies with ISIN: {consolidated_df['isin'].notna().sum()}")
print(
    f"   - Companies with industry_sector: {consolidated_df['industry_sector'].notna().sum()}"
)
print(
    f"   - Companies with company_status: {consolidated_df['company_status'].notna().sum()}"
)

print("\n4. OUTPUT:")
print(f"   - Location: {companies_staging_path}")
print("   - Format: Parquet (optimized) and CSV (readable)")
print("   - Files created:")
print("     - consolidated_bolsa.parquet")
print("     - consolidated_bolsa.csv")

print("\n5. KEY FINDINGS:")
print(f"   - Companies listed on B3: {consolidated_df['asset_name'].notna().sum()}")
print(f"   - Companies with ISIN codes: {consolidated_df['isin'].notna().sum()}")

print("\n6. COMPANY STATUS DISTRIBUTION (CVM):")
if consolidated_df["company_status"].notna().sum() > 0:
    status_counts = consolidated_df["company_status"].value_counts()
    for status, count in status_counts.items():
        pct = (count / len(consolidated_df)) * 100
        print(f"     - {status:15s}: {count:4d} ({pct:5.1f}%)")

print("\n" + "=" * 70)

DATALAKE INVESTIGATION SUMMARY - BOLSA (STOCK EXCHANGE) COMPANIES

1. SOURCE DATASETS:
   - CVM Companies Registration: 2654 records (all)
   - CVM BOLSA filter: 464 companies registered for stock exchange
   - B3 Company Details: 29032 records (all)

2. CONSOLIDATED DATASET (BOLSA):
   - Total unique companies: 464
   - Columns: 9

3. DATA COVERAGE:
   - Companies with code_cvm: 464
   - Companies with company_name: 464
   - Companies with trading_name: 463
   - Companies with asset_name: 367
   - Companies with ISIN: 301
   - Companies with industry_sector: 367
   - Companies with company_status: 464

4. OUTPUT:
   - Location: /home/wilson/snap/brasa/db/staging/companies
   - Format: Parquet (optimized) and CSV (readable)
   - Files created:
     - consolidated_bolsa.parquet
     - consolidated_bolsa.csv

5. KEY FINDINGS:
   - Companies listed on B3: 367
   - Companies with ISIN codes: 301

6. COMPANY STATUS DISTRIBUTION (CVM):
     - ATIVO          :  327 ( 70.5%)
     - CANCELADA  

## 7. Industry Sector Classification Mapping

Map B3's hierarchical industry classification to standard taxonomies (GICS, ICB, and custom normalized sectors).


In [31]:
# Extract hierarchical levels from industry sector
# Create a copy for mapping
df_with_mapping = verified_df.copy()

# Parse the 3-level hierarchical structure (only for non-null values)
df_with_mapping[["sector_level1", "sector_level2", "sector_level3"]] = df_with_mapping[
    "industry_sector"
].str.split(" / ", expand=True, n=2)

print("Industry Sector Hierarchical Structure (BOLSA Companies):")
print(
    f"\nTotal companies with industry_sector data: {df_with_mapping['industry_sector'].notna().sum()}"
)
print(f"\nLevel 1 (Main Sectors): {df_with_mapping['sector_level1'].nunique()} unique")
print(df_with_mapping["sector_level1"].value_counts())
print(f"\nLevel 2 (Subsectors): {df_with_mapping['sector_level2'].nunique()} unique")
print(f"\nLevel 3 (Segments): {df_with_mapping['sector_level3'].nunique()} unique")

Industry Sector Hierarchical Structure (BOLSA Companies):

Total companies with industry_sector data: 367

Level 1 (Main Sectors): 12 unique
sector_level1
Consumo Cíclico                    87
Financeiro                         63
Utilidade Pública                  55
Bens Industriais                   53
Consumo não Cíclico                29
Materiais Básicos                  25
Saúde                              18
Tecnologia da Informação           13
Petróleo. Gás e Biocombustíveis    12
Outros                              4
Não Classificados                   4
Comunicações                        4
Name: count, dtype: int64

Level 2 (Subsectors): 43 unique

Level 3 (Segments): 85 unique


In [32]:
# Create mappings to standard industry classifications
# GICS (Global Industry Classification Standard) - 11 sectors
gics_mapping = {
    "Financeiro": "Financials",
    "Bens Industriais": "Industrials",
    "Materiais Básicos": "Materials",
    "Consumo Cíclico": "Consumer Discretionary",
    "Consumo não Cíclico": "Consumer Staples",
    "Utilidade Pública": "Utilities",
    "Petróleo. Gás e Biocombustíveis": "Energy",
    "Saúde": "Health Care",
    "Tecnologia da Informação": "Information Technology",
    "Comunicações": "Communication Services",
    "Imobiliário": "Real Estate",
    "Construção e Transporte": "Industrials",
    "Outros": "Other",
    "Não Classificados": "Unclassified",
    "Carga Inicial": "Unclassified",
}

# ICB (Industry Classification Benchmark) - 10 sectors
icb_mapping = {
    "Financeiro": "Financials",
    "Bens Industriais": "Industrials",
    "Materiais Básicos": "Basic Materials",
    "Consumo Cíclico": "Consumer Goods",
    "Consumo não Cíclico": "Consumer Services",
    "Utilidade Pública": "Utilities",
    "Petróleo. Gás e Biocombustíveis": "Oil & Gas",
    "Saúde": "Health Care",
    "Tecnologia da Informação": "Technology",
    "Comunicações": "Telecommunications",
    "Imobiliário": "Real Estate",
    "Construção e Transporte": "Industrials",
    "Outros": "Other",
    "Não Classificados": "Unclassified",
    "Carga Inicial": "Unclassified",
}

# Apply GICS mapping
df_with_mapping["gics_sector"] = df_with_mapping["sector_level1"].map(gics_mapping)

# Apply ICB mapping
df_with_mapping["icb_sector"] = df_with_mapping["sector_level1"].map(icb_mapping)

print("Mapping Results:")
print(f"\nGICS Sectors ({df_with_mapping['gics_sector'].nunique()} unique):")
print(df_with_mapping["gics_sector"].value_counts())
print(f"\nICB Sectors ({df_with_mapping['icb_sector'].nunique()} unique):")
print(df_with_mapping["icb_sector"].value_counts())

Mapping Results:

GICS Sectors (12 unique):
gics_sector
Consumer Discretionary    87
Financials                63
Utilities                 55
Industrials               53
Consumer Staples          29
Materials                 25
Health Care               18
Information Technology    13
Energy                    12
Other                      4
Unclassified               4
Communication Services     4
Name: count, dtype: int64

ICB Sectors (12 unique):
icb_sector
Consumer Goods        87
Financials            63
Utilities             55
Industrials           53
Consumer Services     29
Basic Materials       25
Health Care           18
Technology            13
Oil & Gas             12
Other                  4
Unclassified           4
Telecommunications     4
Name: count, dtype: int64


In [33]:
# Create a more granular subsector mapping
# Combining Level 1 + Level 2 for better granularity
subsector_mapping = {
    # Financeiro
    ("Financeiro", "Intermediários Financeiros"): "Banking",
    ("Financeiro", "Previdência e Seguros"): "Insurance & Pension Funds",
    ("Financeiro", "Exploração de Imóveis"): "Real Estate",
    ("Financeiro", "Securitizadoras de Recebíveis"): "Asset Management & Investment",
    ("Financeiro", "Serviços Financeiros Diversos"): "Financial Services",
    ("Financeiro", "Holdings Diversificadas"): "Holding Companies",
    ("Financeiro", "Serviços Diversos"): "Financial Services",
    # Bens Industriais
    ("Bens Industriais", "Material de Transporte"): "Transportation Equipment",
    ("Bens Industriais", "Máquinas e Equipamentos"): "Machinery & Equipment",
    ("Bens Industriais", "Transporte"): "Transportation Services",
    ("Bens Industriais", "Construção e Engenharia"): "Construction",
    ("Bens Industriais", "Serviços"): "Industrial Services",
    ("Bens Industriais", "Comércio"): "Trade & Distribution",
    ("Bens Industriais", "Tecnologia da Informação"): "Technology",
    # Materiais Básicos
    ("Materiais Básicos", "Siderurgia e Metalurgia"): "Steel & Metals",
    ("Materiais Básicos", "Mineração"): "Mining",
    ("Materiais Básicos", "Químicos"): "Chemicals",
    ("Materiais Básicos", "Madeira e Papel"): "Timber & Pulp",
    ("Materiais Básicos", "Embalagens"): "Packaging",
    ("Materiais Básicos", "Materiais Diversos"): "Other Materials",
    # Consumo Cíclico
    ("Consumo Cíclico", "Automóveis e Motocicletas"): "Automobiles",
    ("Consumo Cíclico", "Comércio"): "Retail",
    ("Consumo Cíclico", "Construção Civil"): "Real Estate Development",
    ("Consumo Cíclico", "Hoteis e Restaurantes"): "Hotels & Restaurants",
    ("Consumo Cíclico", "Tecidos. Vestuário e Calçados"): "Apparel & Footwear",
    ("Consumo Cíclico", "Utilidades Domésticas"): "Home Appliances & Furnishings",
    ("Consumo Cíclico", "Viagens e Lazer"): "Leisure & Travel",
    ("Consumo Cíclico", "Diversos"): "Diversified Consumer",
    # Consumo não Cíclico
    ("Consumo não Cíclico", "Agropecuária"): "Agriculture",
    ("Consumo não Cíclico", "Alimentos Processados"): "Food & Beverage",
    ("Consumo não Cíclico", "Bebidas"): "Beverages",
    ("Consumo não Cíclico", "Comércio e Distribuição"): "Retail & Distribution",
    (
        "Consumo não Cíclico",
        "Produtos de Uso Pessoal e de Limpeza",
    ): "Personal Care & Hygiene",
    ("Consumo não Cíclico", "Diversos"): "Diversified Consumer",
    # Utilidade Pública
    ("Utilidade Pública", "Energia Elétrica"): "Electric Utilities",
    ("Utilidade Pública", "Gás"): "Gas Utilities",
    ("Utilidade Pública", "Água e Saneamento"): "Water & Sewer",
    # Petróleo
    (
        "Petróleo. Gás e Biocombustíveis",
        "Petróleo. Gás e Biocombustíveis",
    ): "Oil, Gas & Biofuels",
    (
        "Petróleo. Gás e Biocombustíveis",
        "Equipamentos e Serviços",
    ): "Oil & Gas Equipment & Services",
    (
        "Petróleo. Gás e Biocombustíveis",
        "Exploração. Refino e Distribuição",
    ): "Oil & Gas Exploration & Production",
    (
        "Petróleo. Gás e Biocombustíveis",
        "Distribuição de Combustíveis",
    ): "Fuel Distribution",
    # Saúde
    ("Saúde", "Medicamentos e Outros Produtos"): "Pharmaceuticals",
    ("Saúde", "Comércio e Distribuição"): "Pharmaceutical Distribution",
    ("Saúde", "Equipamentos"): "Medical Equipment",
    ("Saúde", "Serv.Méd.Hospit..Análises e Diagnósticos"): "Healthcare Services",
    # Tecnologia da Informação
    ("Tecnologia da Informação", "Computadores e Equipamentos"): "Computer Hardware",
    ("Tecnologia da Informação", "Programas e Serviços"): "Software & IT Services",
    # Comunicações
    ("Comunicações", "Telecomunicações"): "Telecommunications",
    ("Comunicações", "Mídia"): "Media & Entertainment",
    # Construção e Transporte
    ("Construção e Transporte", "Transporte"): "Transportation Services",
    ("Construção e Transporte", "Construção e Engenharia"): "Construction",
}

# Apply subsector mapping
df_with_mapping["subsector"] = df_with_mapping.apply(
    lambda row: subsector_mapping.get(
        (row["sector_level1"], row["sector_level2"]), "Other"
    ),
    axis=1,
)

print("Subsector Mapping:")
print(f"Unique subsectors: {df_with_mapping['subsector'].nunique()}")
print("\nTop subsectors by company count:")
print(df_with_mapping["subsector"].value_counts().head(20))

Subsector Mapping:
Unique subsectors: 43

Top subsectors by company count:
subsector
Other                        105
Electric Utilities            46
Banking                       26
Real Estate Development       25
Apparel & Footwear            19
Transportation Services       18
Real Estate                   18
Retail                        15
Food & Beverage               14
Machinery & Equipment         13
Oil, Gas & Biofuels           12
Software & IT Services        11
Diversified Consumer          11
Healthcare Services           10
Steel & Metals                10
Transportation Equipment       8
Insurance & Pension Funds      8
Water & Sewer                  7
Construction                   7
Leisure & Travel               6
Name: count, dtype: int64


In [34]:
# Save enriched dataset with classifications
# Select final columns for output
final_mapped_df = df_with_mapping[
    [
        "cnpj",
        "company_name",
        "trading_name",
        "asset_name",
        "company_status",
        "isin",
        "industry_sector",  # Original B3 hierarchical classification
        "sector_level1",  # B3 Main sector
        "sector_level2",  # B3 Subsector
        "sector_level3",  # B3 Industry segment
        "gics_sector",  # GICS standard (11 sectors)
        "icb_sector",  # ICB standard (10 sectors)
        "subsector",  # Normalized English subsector
    ]
].copy()

# Save as parquet with mapping
consolidated_mapped_parquet = os.path.join(
    companies_staging_path, "consolidated_with_classification.parquet"
)
final_mapped_df.to_parquet(consolidated_mapped_parquet, index=False)

print("✓ Consolidated dataset with classification mapping saved")
print(f"  Location: {consolidated_mapped_parquet}")
print(f"  File size: {os.path.getsize(consolidated_mapped_parquet) / 1024:.2f} KB")
print(f"  Records: {len(final_mapped_df)}")
print(f"  Columns: {len(final_mapped_df.columns)}")

# Also save a CSV version
consolidated_mapped_csv = os.path.join(
    companies_staging_path, "consolidated_with_classification.csv"
)
final_mapped_df.to_csv(consolidated_mapped_csv, index=False)
print("\n✓ CSV version also saved: consolidated_with_classification.csv")

# Display sample with mappings
print("\n--- Sample records with industry classification mapping ---")
display(
    final_mapped_df[
        ["company_name", "industry_sector", "gics_sector", "icb_sector", "subsector"]
    ].head(10)
)

KeyError: "['cnpj'] not in index"